# 🎬 Video Generation (📦 Archive)

> ## ⚠️ 아카이브 노트북 — 지금은 실행되지 않아요
> OpenAI의 **Sora 2 모델(`sora-2`, `sora-2-pro`)과 Videos API는 2026년 9월 24일에 종료**되었고, 현재 1:1로 대체되는 OpenAI API는 없습니다.
>
> 이 노트북은 2025년 11월 수업 당시의 **코드와 실행 결과를 그대로 보존한 기록용**입니다.
> 그래도 *작업(job) 생성 → 상태 polling → 결과 다운로드* 로 이어지는 **비동기 API 패턴**은 다른 영상/음성 생성 API에서도 거의 똑같이 쓰이니까, 흐름 참고용으로 봐주세요.
>
> 당시 생성된 결과 영상은 [`data/video/`](data/video/) 폴더에 있습니다.

## 환경 준비 및 API 키 설정

In [ ]:
%pip install -q -U openai

In [ ]:
import os

OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY")
if not OPENAI_API_KEY:
    try:
        from google.colab import userdata  # type: ignore
        OPENAI_API_KEY = userdata.get("OPENAI_API_KEY")
    except ImportError:
        pass  # Colab이 아닌 환경
    except Exception:
        print("Colab 메뉴의 보안 비밀(열쇠 아이콘)에 OPENAI_API_KEY가 저장되어 있는지, 노트북 액세스가 켜져 있는지 확인해주세요.")

if not OPENAI_API_KEY:
    from getpass import getpass
    OPENAI_API_KEY = getpass("OPENAI_API_KEY를 입력하세요: ")

os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
print("✅ API 키 설정 완료")

In [ ]:
base_dirs = [
    ("data", ["image", "audio", "video"]),
    ("results", ["image", "audio", "video"]),
]

for root, subfolders in base_dirs:
    for sub in subfolders:
        path = os.path.join(root, sub)
        os.makedirs(path, exist_ok=True)
        print(f"ready: {path}")

In [ ]:
from openai import OpenAI, AsyncOpenAI

client = OpenAI()
async_client = AsyncOpenAI()
print("✅ 클라이언트 생성 완료")


✅ 클라이언트 생성 완료


## 1) Overview of OpenAI Sora 2
텍스트 또는 이미지 입력을 받아 동영상을 생성하는 모델입니다.

Video API에서 제공하는 기능

- **Create video**: 텍스트/이미지 프롬프트로 비디오 생성 작업을 시작
- **Get video status**: 생성 작업의 상태(queued, in_progress, completed, failed)를 조회
- **Download video**: 생성 완료된 비디오(MP4)를 다운로드
- **List videos**: 사용자가 생성한 비디오 목록 확인
- **Delete videos**: 더 이상 필요 없는 비디오 삭제

이 과정은 비동기(asynchronous)로 동작합니다.

1. `POST /videos` → 작업(job)을 서버에 등록 (MP4가 바로 오지 않음)
2. `GET /videos/{video_id}` 또는 Webhook → 작업 상태를 확인
3. 상태가 `completed`가 되면
4. `GET /videos/{video_id}/content` → 최종 비디오 파일(MP4)을 다운로드

## 2) Sora 모델 라인업 (sora-2 vs sora-2-pro)
| 모델 | 특징 | 추천 시나리오 |
| --- | --- | --- |
| `sora-2` | 빠른 생성 속도, 실험/컨셉 초안에 적합 | 아이디어 스케치, SNS 콘텐츠 제작, 반복 실험 |
| `sora-2-pro` | 고품질, 느린 생성 시간, 높은 비용 | 마케팅/시네마틱, 고해상도 최종본 |


## 3) 효과적인 프롬프팅 가이드
좋은 비디오를 얻으려면, 가능하면 다음 요소들을 함께 넣어주는 것이 좋습니다.

- **Shot type**: 와이드샷, 클로즈업, 트래킹샷 등
- **Subject**: 무엇을 찍는지 (아이, 캐릭터, 자동차…)
- **Action**: 무엇을 하고 있는지 (걷는다, 날아간다, 손을 흔든다…)
- **Setting**: 배경/장소 (공원, 대학교 캠퍼스, 실내 카페…)
- **Lighting**: 조명/시간대 (저녁 노을, 아침 햇살, 네온사인 등)


**예시:**
- “넓은 공원에서 붉은 연을 날리는 아이를 담은 **와이드샷**, **저녁 노을**, 카메라가 **천천히 위로 이동**한다.”
- “나무 테이블 위 **뜨거운 커피잔을 클로즈업**, **블라인드 사이로 들어오는 아침 햇빛**, **얕은 심도**의 부드러운 배경.”

**추가 자료**
- https://cookbook.openai.com/examples/sora/sora2_prompting_guide

## 4) 비디오 생성 시작하기 (Create)
아래 예시는 `sora-2` 모델을 사용해 **간단한 비디오 생성 작업**을 등록하는 코드입니다.

In [ ]:
from openai import OpenAI
client = OpenAI()

video_job = client.videos.create(
    model="sora-2",
    prompt="서울 명동 거리를 달리는 오토바이를 탄 고양이를 영화 같은 느낌으로 보여주는 비디오",
    seconds= "4",        # 길이 지정, "4", "8", "12"
    size="1280x720",  # 해상도 지정, 1280x720, 720x1280
)

print("Video generation job created:")
print("  id     :", video_job.id)
print("  status :", video_job.status)
print("  model  :", video_job.model)


Video generation job created:
  id     : video_69282b68b9688190815a66f6d5b87c7c07c606296a80a802
  status : queued
  model  : sora-2


**응답 예시**

```json
{
  "id": "video_68d7512d07848190b3e45da0ecbebcde004da08e1e0678d5",
  "object": "video",
  "created_at": 1758941485,
  "status": "queued",
  "model": "sora-2",
  "progress": 0,
  "seconds": "8",
  "size": "1280x720"
}
```

## 5) 비동기 흐름 상세 설명 (1~4단계)

1. **작업 등록 (`POST /videos`)**  
   - `client.videos.create(...)` 호출 → `id`, `status` 등을 포함한 **작업(job) 객체** 반환
   - 예: `status="queued"` → “작업이 대기열에 들어갔다”는 의미

2. **상태 확인 (`GET /videos/{video_id}` 또는 Webhook)**  
   - Polling: 주기적으로 `client.videos.retrieve(video_id)` 호출
   - Webhook: 작업이 끝나면 OpenAI가 우리가 등록한 URL로 알림을 보내줌

3. **완료 상태(`completed`) 감지**  
   - 상태: `queued` → `in_progress` → `completed` 
   - `completed`가 된 시점에서 **“영상 생성 완료”**

4. **최종 MP4 다운로드 (`/content`)**  
   - `client.videos.download_content(video_id, variant="video")`를 호출하여
     비디오 파일(MP4)을 받아와 로컬에 저장 가능

**상태 확인**

In [ ]:
video_job_status= client.videos.retrieve(video_job.id)
print(video_job_status)

**완료 상태 감지 및 비디오 다운로드**

In [ ]:
if video_job.status == "completed":
    content = client.videos.download_content(video_job.id, variant="video")
    content.write_to_file("results/video/bike_cat_video.mp4")
    print("Saved to bike_cat_video.mp4")

## 6) `create_and_poll`

OpenAI Python SDK는 위의 과정을 편하게 해주는 **`create_and_poll` 메서드**를 제공

- 내부 순서
1. `create` 로 작업을 생성
2. 상태가 `completed` 혹은 `failed`가 될 때까지 주기적으로 상태 조회

In [ ]:
from openai import AsyncOpenAI

client = AsyncOpenAI()

async def main() -> None:
    video = await client.videos.create_and_poll(
        model="sora-2",
        prompt="한강을 수영하는 비둘기를 애니메이션 같은 느낌으로 보여주는 비디오",
        seconds="4",
    )

    if video.status == "completed":
        print("Video successfully completed: ", video)
    else:
        print("Video creation failed. Status: ", video.status)
    
    return video

video = await main()

print("Final status:", video.status)

if video.status == "completed":
    content = await client.videos.download_content(video.id, variant="video")
    content.write_to_file("results/video/pigeon_video.mp4")
    print("Saved to pigeon_video.mp4")


Video successfully completed:  Video(id='video_69283c97d6f48190bc2f0e359989ef9d0d60e8f0a9ad9e3f', completed_at=1764245635, created_at=1764244631, error=None, expires_at=1764249235, model='sora-2', object='video', progress=100, prompt='한강을 수영하는 비둘기를 애니메이션 같은 느낌으로 보여주는 비디오', remixed_from_video_id=None, seconds='4', size='720x1280', status='completed')
Final status: completed
Saved to pigeon_video.mp4


▶️ 당시 생성된 영상: [`pigeon_video.mp4`](data/video/pigeon_video.mp4) — *한강을 수영하는 비둘기* 🕊️

## 7) Polling + 진행률 표시 + 비디오 다운로드 예시

Polling 루프를 구현
 - 상태 변화(`queued`, `in_progress`, `completed`)를 확인한다.
 - `progress` 필드를 이용해서 진행률을 표시
 - 작업이 완료되면 MP4 파일로 저장

In [ ]:
import sys, asyncio
from pathlib import Path

from openai import AsyncOpenAI

client = AsyncOpenAI()
bar_length = 30

async def poll_video():
    video_job = await client.videos.create(
        model="sora-2",
        prompt="서울의 밤하늘에서 서서히 줌인하는 드론 샷, 네온사인과 자동차 불빛이 반짝이는 비디오",
    )

    print("Video generation started:")
    print("  id     :", video_job.id)
    print("  status :", video_job.status)

    progress = getattr(video_job, "progress", 0)

    while video_job.status in ("queued", "in_progress"):
        video_job = await client.videos.retrieve(video_job.id)
        progress = getattr(video_job, "progress", 0)

        filled_length = int((progress / 100) * bar_length)
        bar = "=" * filled_length + "-" * (bar_length - filled_length)
        status_text = "Queued" if video_job.status == "queued" else "Processing"

        sys.stdout.write(f"\r{status_text}: [{bar}] {progress:.1f}%")
        sys.stdout.flush()

        await asyncio.sleep(2)

    sys.stdout.write("\n")

    if video_job.status == "failed":
        message = getattr(getattr(video_job, "error", None), "message", "Video generation failed")
        print(message)
        return

    print("✅ Video generation completed!:", video_job)
    print("id:", video_job.id)

    print("Downloading video...")
    Path("results/video/").mkdir(parents=True, exist_ok=True)
    content = await client.videos.download_content(video_job.id, variant="video")
    content.write_to_file("results/video/seoul_droneshot_video.mp4")
    print("🎥 Saved to seoul_droneshot_video.mp4")

await poll_video()

Video generation started:
  id     : video_69282ee4f55881908b4c374204580b360c50d12b786db9ad
  status : queued
Processing: [==============================] 100.0%
✅ Video generation completed!: Video(id='video_69282ee4f55881908b4c374204580b360c50d12b786db9ad', completed_at=1764241188, created_at=1764241125, error=None, expires_at=1764244788, model='sora-2', object='video', progress=100, prompt='서울의 밤하늘에서 서서히 줌인하는 드론 샷, 네온사인과 자동차 불빛이 반짝이는 비디오', remixed_from_video_id=None, seconds='4', size='720x1280', status='completed')
id: video_69282ee4f55881908b4c374204580b360c50d12b786db9ad
🎥 Saved to sora2_demo_video.mp4


## 8) 이미지 레퍼런스를 사용한 비디오 생성
**입력 이미지 한 장을 비디오의 첫 프레임**으로 사용하여 해당 스타일/구도를 유지한 채로 장면을 전개 가능

- 입력 이미지: [`data/image/chito_1280x720.png`](data/image/chito_1280x720.png) (아주대학교 마스코트 '치토')

In [ ]:
from io import BytesIO
from openai import OpenAI
client = OpenAI()

input_image_path = "data/image/chito_1280x720.png"

with open(input_image_path, "rb") as f:
    image_bytes = f.read()

video_job_chito = client.videos.create_and_poll(
    model="sora-2",
    prompt="캐릭터들이 대학 캠퍼스 산책로를 천천히 걷는다, 밝고 상쾌한 애니메이션 분위기의 와이드샷.",
    input_reference=("chito.png", BytesIO(image_bytes), "image/png"),
    size="1280x720",
    seconds="4",
)

print("Final status:", video_job_chito.status)

if video_job_chito.status == "completed":
    content = client.videos.download_content(video_job_chito.id, variant="video")
    content.write_to_file("results/video/chito_video.mp4")
    print("Saved to chito_video.mp4")

Final status: completed
Saved to chito_video.mp4


▶️ 당시 생성된 영상: [`chito_video.mp4`](data/video/chito_video.mp4)

## 9) Remix: 기존 비디오를 수정해서 재생성
**Remix** 기능을 통해 **색감, 분위기, 일부 요소만 바꿔서 재생성** 가능

아이디어:
- “캐릭터들의 색깔을 빨강색으로 바꿔줘”
- “캐릭터들이 달려가는 것으로 바꿔줘”

In [ ]:
previous_video_id = video_job_chito.id

remixed_job = client.videos.remix(
    video_id=previous_video_id,
    prompt="캐릭터들의 색깔을 빨강색으로 바꿔줘",
)

print("Remix status:", remixed_job.status)

Remix status: queued


In [ ]:
import time

while remixed_job.status in ("queued", "in_progress"):
    time.sleep(2)  # 2초 간격
    remixed_job = client.videos.retrieve(remixed_job.id)

if remixed_job.status == "completed":
    content = client.videos.download_content(remixed_job.id, variant="video")
    content.write_to_file("results/video/remixed_video.mp4")
    print("Saved to remixed_video.mp4")
else:
    print("Remix failed:", remixed_job.status)

Saved to remixed_video.mp4


▶️ 당시 생성된 영상: [`remixed_video.mp4`](data/video/remixed_video.mp4) — *캐릭터들의 색깔을 빨강색으로 바꿔줘*

## 10) 기타

API는 다음과 같은 콘텐츠 제한을 적용합니다:

- 18세 미만 관람가에 적합한 콘텐츠만 생성할 수 있습니다.
(향후 이 제한을 비활성화할 수 있는 설정이 제공될 예정입니다.)

- 저작권이 있는 캐릭터나 저작권 음악은 거절됩니다.

- 실존 인물(공인 포함)은 생성할 수 없습니다.

- 사람의 얼굴이 포함된 입력 이미지는 현재 사용이 불가능합니다.

- 프롬프트, 참고 이미지, 대본(Transcript)을 작성할 때
위 규칙을 준수하지 않으면 생성이 실패할 수 있으니 주의하세요.

### 참고 자료
- https://platform.openai.com/docs/guides/video-generation
- https://platform.openai.com/docs/api-reference